# SemEval-2026 Task 13 — Subtask A: GraphCodeBERT

**Binary Classification** — Detect machine-generated vs human-written code.

### Model: `microsoft/graphcodebert-base`
- Pre-trained with data flow information → structure-aware representations
- Same RoBERTa architecture as CodeBERT → drop-in replacement
- Hypothesis: DFG-aware pretraining improves code understanding

### Setup
- **GPU:** Kaggle Tesla T4 (16 GB VRAM)
- **Data:** 10% stratified subsample of training set
- **Metric:** Macro F1 (competition metric)

Set runtime to `GPU T4 x2` or `GPU T4` in Kaggle settings.

In [1]:
# ============================================================
# Cell 1: Environment Setup
# ============================================================
!pip install transformers==4.45.0 huggingface_hub datasets scikit-learn accelerate pyarrow fastparquet tqdm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 70.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 84.7 MB/s eta 0:00:00


In [2]:
# ============================================================
# Cell 2: Imports
# ============================================================
import os
os.environ["WANDB_DISABLED"] = "true"

import gc
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from datasets import Dataset, load_dataset
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
)
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    f1_score,
)
import warnings
warnings.filterwarnings("ignore")

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    vram = getattr(props, 'total_memory', getattr(props, 'total_mem', 0))
    print(f"VRAM: {vram / 1e9:.1f} GB")

2026-04-05 07:01:19.176009: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775372479.362045      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775372479.413159      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775372479.872560      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775372479.872583      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775372479.872586      24 computation_placer.cc:177] computation placer alr

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


## Configuration

In [3]:
# ============================================================
# Cell 3: Configuration
# ============================================================

# --- Model ---
MODEL_NAME = "microsoft/graphcodebert-base"

# --- Data paths (Kaggle) ---
TRAIN_PATH = (
    "/kaggle/input/datasets/daniilor/semeval-2026-task13/"
    "SemEval-2026-Task13/task_a/task_a_training_set_1.parquet"
)
VAL_PATH = (
    "/kaggle/input/datasets/daniilor/semeval-2026-task13/"
    "SemEval-2026-Task13/task_a/task_a_validation_set.parquet"
)
TEST_PATH = (
    "/kaggle/input/datasets/daniilor/semeval-2026-task13/"
    "SemEval-2026-Task13/task_a/task_a_test_set_sample.parquet"
)

# --- Subsampling (10% of ~500K dataset) ---
TRAIN_SAMPLE_SIZE = 50000       # 10% of 500K
VAL_SAMPLE_SIZE   = 12500       # 25% of train sample
RANDOM_SEED       = 42

# --- Validation Mode ---
USE_LOLO_VALIDATION = True      # Implement Leave-One-Language-Out
HOLDOUT_LANGUAGE    = "c++"     # E.g. 'c++', 'python', 'java'

# --- Training hyperparameters ---
MAX_LENGTH         = 512         # Doubled via gradient checkpointing
BATCH_SIZE         = 16
GRAD_ACCUM_STEPS   = 2           # Effective batch = 32
LEARNING_RATE      = 2e-5
NUM_EPOCHS         = 10
WEIGHT_DECAY       = 0.01
WARMUP_RATIO       = 0.1
LABEL_SMOOTHING    = 0.0         # Handled entirely by Focal Loss
EARLY_STOPPING_PATIENCE = 5

# --- Output ---
OUTPUT_DIR = "/kaggle/working/results_graphcodebert_taskA"

print(f"Model: {MODEL_NAME}")
print(f"Max length: {MAX_LENGTH} (with check-pointing)")
print(f"LOLO Validation Mode: {USE_LOLO_VALIDATION} (Holdout: {HOLDOUT_LANGUAGE})")
print(f"Train sample: {TRAIN_SAMPLE_SIZE} (10% of dataset), Val sample: {VAL_SAMPLE_SIZE}")
print(f"Effective batch size: {BATCH_SIZE * GRAD_ACCUM_STEPS}")
print(f"Learning rate: {LEARNING_RATE}")

Model: microsoft/graphcodebert-base
Max length: 512 (with check-pointing)
LOLO Validation Mode: True (Holdout: c++)
Train sample: 50000 (10% of dataset), Val sample: 12500
Effective batch size: 32
Learning rate: 2e-05


## GraphCodeBERT Trainer Class

Follows the baseline `CodeClassifierTrainer` structure with:
- GraphCodeBERT backbone (`microsoft/graphcodebert-base`)
- Macro F1 as the primary metric (competition metric)
- Cosine LR scheduler + early stopping
- fp16 mixed precision for T4

In [4]:
# ============================================================
# Cell 4: GraphCodeBERT Custom Trainer & Model
# ============================================================

from transformers import RobertaModel, PreTrainedModel
import torch.nn.functional as F

# ------------------------------------------------------------------
# Custom Model: Multi-Sample Dropout
# ------------------------------------------------------------------
from transformers import RobertaConfig

class GraphCodeBERTMultiDropModel(PreTrainedModel):
    """
    Wraps GraphCodeBERT inside a multi-sample dropout head to improve generalization
    and stability, especially useful for long texts and small learning rates.
    """
    config_class = RobertaConfig # Crucial line to enable .from_pretrained
    supports_gradient_checkpointing = True
    
    @classmethod
    def _can_set_experts_implementation(cls):
        # Hotfix for Hugging Face issue #33618 within Jupyter Notebooks
        return False
    
    def __init__(self, config, num_labels=2, num_dropouts=5):
        super().__init__(config)
        self.num_labels = num_labels
        self.roberta = RobertaModel(config, add_pooling_layer=False)
        self.dense = nn.Linear(config.hidden_size, config.hidden_size)
        self.dropouts = nn.ModuleList([nn.Dropout(p=0.1 + (i * 0.05)) for i in range(num_dropouts)])
        self.out_proj = nn.Linear(config.hidden_size, num_labels)
        self.post_init()

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        outputs = self.roberta(input_ids, attention_mask=attention_mask, **kwargs)
        sequence_output = outputs[0][:, 0, :]  # CLS token representation
        sequence_output = self.dense(sequence_output)
        sequence_output = torch.tanh(sequence_output)

        # Compute multiple dropouts and average the logits
        logits = torch.mean(
            torch.stack([self.out_proj(dropout(sequence_output)) for dropout in self.dropouts]),
            dim=0
        )

        loss = None
        if labels is not None:
             loss = nn.CrossEntropyLoss()(logits, labels.view(-1))
        
        from transformers.modeling_outputs import SequenceClassifierOutput
        return SequenceClassifierOutput(
            loss=loss,
            logits=logits,
            hidden_states=outputs.hidden_states,
            attentions=outputs.attentions,
        )

# ------------------------------------------------------------------
# Custom Trainer: Focal Loss
# ------------------------------------------------------------------
class FocalLossTrainer(Trainer):
    """
    Overrides standard CrossEntropy to FocalLoss, helping the model
    focus on hard 'borderline' examples between complex AI and Human code.
    """
    def __init__(self, *args, alpha=1.0, gamma=2.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.alpha = alpha
        self.gamma = gamma

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels", None)
        outputs = model(**inputs)
        logits = outputs.get("logits")
        
        if labels is not None:
            ce_loss = F.cross_entropy(logits, labels.view(-1), reduction="none")
            pt = torch.exp(-ce_loss)
            focal_loss = (self.alpha * (1 - pt) ** self.gamma * ce_loss).mean()
        else:
            focal_loss = outputs.get("loss")
            
        return (focal_loss, outputs) if return_outputs else focal_loss


class GraphCodeBERTTrainerA:
    def __init__(self, model_name=MODEL_NAME, max_length=MAX_LENGTH):
        self.model_name = model_name
        self.max_length = max_length
        self.tokenizer = None
        self.model = None
        self.num_labels = None
        self.tag = model_name.split("/")[-1]

    def load_and_prepare_data(
        self,
        sample_size=TRAIN_SAMPLE_SIZE,
        val_sample_size=VAL_SAMPLE_SIZE,
        random_seed=RANDOM_SEED,
    ):
        try:
            global USE_LOLO_VALIDATION
            df = pd.read_parquet(TRAIN_PATH)
            
            if 'code' not in df.columns or 'label' not in df.columns:
                raise ValueError("Dataset must contain 'code' and 'label' columns")

            df = df.dropna(subset=['code', 'label'])
            df['label'] = df['label'].astype(int)
            
            # --- Leave-One-Language-Out Configuration ---
            if USE_LOLO_VALIDATION and 'language' in df.columns:
                print(f"[{self.tag}] Using LOLO Validation. Holding out: {HOLDOUT_LANGUAGE}")
                
                # Force case-insensitive match
                df['language_lower'] = df['language'].str.lower()
                
                # Split
                train_mask = df['language_lower'] != HOLDOUT_LANGUAGE
                val_mask = df['language_lower'] == HOLDOUT_LANGUAGE
                
                train_df = df[train_mask]
                val_df = df[val_mask]
                
                if len(val_df) == 0:
                    print(f"Warning: Holdout language {HOLDOUT_LANGUAGE} not found. Falling back to default.")
                    USE_LOLO_VALIDATION = False
                else:
                    if sample_size and sample_size < len(train_df):
                        train_df = train_df.groupby('label', group_keys=False).apply(
                            lambda x: x.sample(n=max(1, int(sample_size * len(x) / len(train_df))), random_state=random_seed)
                        )
                    if val_sample_size and val_sample_size < len(val_df):
                        val_df = val_df.groupby('label', group_keys=False).apply(
                            lambda x: x.sample(n=max(1, int(val_sample_size * len(x) / len(val_df))), random_state=random_seed)
                        )
                    df = train_df.reset_index(drop=True)
                    val_df = val_df.reset_index(drop=True)

            if not USE_LOLO_VALIDATION or 'language' not in df.columns:
                print(f"[{self.tag}] Using stratified random validation split.")
                # Stratified sample
                if sample_size is not None and sample_size < len(df):
                    df = df.groupby('label', group_keys=False).apply(
                        lambda x: x.sample(
                            n=max(1, int(sample_size * len(x) / len(df))),
                            random_state=random_seed,
                        )
                    ).reset_index(drop=True)
                
                # Standard val set
                val_df = pd.read_parquet(VAL_PATH)
                val_df = val_df.dropna(subset=['code', 'label'])
                val_df['label'] = val_df['label'].astype(int)

                if val_sample_size is not None and val_sample_size < len(val_df):
                    val_df = val_df.groupby('label', group_keys=False).apply(
                        lambda x: x.sample(
                            n=max(1, int(val_sample_size * len(x) / len(val_df))),
                            random_state=random_seed,
                        )
                    ).reset_index(drop=True)

            self.num_labels = df['label'].nunique()
            print(f"[{self.tag}] Train: {len(df)},  Val: {len(val_df)}")
            return df, val_df

        except Exception as e:
            print(f"[{self.tag}] Error loading dataset: {e}")
            raise

    def initialize_model_and_tokenizer(self):
        print(f"[{self.tag}] Initialising extended model (Multi-Sample Dropout) and tokenizer ...")

        self.tokenizer = RobertaTokenizer.from_pretrained(self.model_name)

        # Using the Multi-Sample Dropout wrapper
        self.model = GraphCodeBERTMultiDropModel.from_pretrained(
            self.model_name,
            num_labels=self.num_labels
        ).to('cuda')

        total = sum(p.numel() for p in self.model.parameters())
        print(f"[{self.tag}] {self.num_labels} labels | params {total:,}")

    def tokenize_function(self, examples):
        return self.tokenizer(
            examples['code'],
            truncation=True,
            padding=True,
            max_length=self.max_length,
            return_tensors="pt",
        )

    def prepare_datasets(self, train_df, val_df):
        print(f"[{self.tag}] Preparing datasets ...")
        train_dataset = Dataset.from_pandas(train_df[['code', 'label']])
        val_dataset   = Dataset.from_pandas(val_df[['code', 'label']])

        train_dataset = train_dataset.map(
            self.tokenize_function, batched=True, remove_columns=['code']
        )
        val_dataset = val_dataset.map(
            self.tokenize_function, batched=True, remove_columns=['code']
        )

        train_dataset = train_dataset.rename_column('label', 'labels')
        val_dataset   = val_dataset.rename_column('label', 'labels')

        return train_dataset, val_dataset

    def compute_metrics(self, eval_pred):
        predictions, labels = eval_pred
        preds = np.argmax(predictions, axis=1)

        accuracy = accuracy_score(labels, preds)
        macro_f1 = f1_score(labels, preds, average='macro', zero_division=0)
        precision, recall, f1_w, _ = precision_recall_fscore_support(
            labels, preds, average='weighted', zero_division=0
        )

        return {
            'accuracy': accuracy,
            'macro_f1': macro_f1,
            'f1': f1_w,
            'precision': precision,
            'recall': recall,
        }

    def train(
        self,
        train_dataset,
        val_dataset,
        output_dir=OUTPUT_DIR,
        num_epochs=NUM_EPOCHS,
        batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE,
    ):
        print(f"[{self.tag}] Starting training ...")

        # Halve the micro-batch size and double accum steps to fit VRAM WITHOUT checkpointing
        safe_batch = max(1, batch_size // 2)
        safe_accum = GRAD_ACCUM_STEPS * 2
        steps_per_epoch = len(train_dataset) // (safe_batch * safe_accum)

        training_args = TrainingArguments(
            output_dir=output_dir,
            num_train_epochs=num_epochs,
            per_device_train_batch_size=safe_batch,
            per_device_eval_batch_size=batch_size * 2,
            gradient_accumulation_steps=safe_accum,
            weight_decay=WEIGHT_DECAY,
            warmup_steps=int(steps_per_epoch * num_epochs * WARMUP_RATIO),
            logging_steps=50,
            eval_strategy="epoch",
            save_strategy="epoch",
            load_best_model_at_end=True,
            metric_for_best_model="macro_f1",
            greater_is_better=True,
            remove_unused_columns=False,
            learning_rate=learning_rate,
            lr_scheduler_type="cosine",
            save_total_limit=2,
            report_to=[],
            fp16=True,
            fp16_full_eval=True,
            gradient_checkpointing=False,
            group_by_length=True,
            dataloader_num_workers=4,
            dataloader_pin_memory=True,
            optim="adamw_torch_fused",
        )

        data_collator = DataCollatorWithPadding(tokenizer=self.tokenizer)

        # Using Custom Focal Loss Trainer
        trainer = FocalLossTrainer(
            model=self.model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
            data_collator=data_collator,
            compute_metrics=self.compute_metrics,
            callbacks=[EarlyStoppingCallback(
                early_stopping_patience=EARLY_STOPPING_PATIENCE
            )],
            alpha=1.0, 
            gamma=2.0
        )

        print(f"[{self.tag}] Config:")
        print(f"  Train: {len(train_dataset)}, Val: {len(val_dataset)}")
        print(f"  LR: {learning_rate}, MaxLen: {self.max_length}")
        print(f"  Gradient Checkpointing: False (Speed optimization via micro-batching)")

        trainer.train()

        trainer.save_model()
        self.tokenizer.save_pretrained(output_dir)
        print(f"[{self.tag}] Training completed. Model saved to {output_dir}")
        return trainer

    def evaluate_model(self, trainer, val_dataset):
        print(f"\n{'='*60}")
        print(f"  EVALUATION — {self.tag}")
        print(f"{'='*60}")

        predictions = trainer.predict(val_dataset)
        y_pred = np.argmax(predictions.predictions, axis=1)
        y_true = predictions.label_ids

        print("\nPer-Class Classification Report:")
        print(classification_report(
            y_true, y_pred,
            target_names=["human", "machine"],
            zero_division=0,
        ))

        macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
        print(f"\n>>> COMPETITION METRIC (Macro F1): {macro_f1:.4f} <<<")
        return predictions

    def run_full_pipeline(
        self,
        output_dir=OUTPUT_DIR,
        num_epochs=NUM_EPOCHS,
        batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        sample_size=TRAIN_SAMPLE_SIZE,
        val_sample_size=VAL_SAMPLE_SIZE,
        random_seed=RANDOM_SEED,
    ):
        try:
            train_df, val_df = self.load_and_prepare_data(
                sample_size=sample_size,
                val_sample_size=val_sample_size,
                random_seed=random_seed,
            )
            self.initialize_model_and_tokenizer()
            train_dataset, val_dataset = self.prepare_datasets(train_df, val_df)
            trainer = self.train(
                train_dataset, val_dataset,
                output_dir=output_dir,
                num_epochs=num_epochs,
                batch_size=batch_size,
                learning_rate=learning_rate,
            )
            self.evaluate_model(trainer, val_dataset)
            print(f"\n[{self.tag}] Pipeline completed successfully!")
            return trainer
        except Exception as e:
            print(f"[{self.tag}] Error in pipeline: {e}")
            raise

print("GraphCodeBERTTrainerA defined.")

GraphCodeBERTTrainerA defined.


## Test Set Evaluation

Runs inference on the test parquet and prints per-category metrics
(seen/unseen language × seen/unseen domain).

In [5]:
# ============================================================
# Cell 5: Test-set evaluation utilities (from baseline)
# ============================================================
from tqdm import tqdm

SEEN_LANGS   = {'c++', 'cpp', 'python', 'java'}
UNSEEN_LANGS = {'go', 'php', 'c#', 'csharp', 'c', 'javascript', 'js'}
SEEN_DOMAINS   = {'algorithmic'}
UNSEEN_DOMAINS = {'research', 'production'}


@torch.no_grad()
def evaluate_on_test(trainer_obj, parquet_path, max_length=256,
                     batch_size=32, device=None):
    """Run inference on test parquet → return DataFrame with 'prediction' column."""
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    tokenizer = trainer_obj.tokenizer
    model = trainer_obj.model
    model.to(device)
    model.eval()

    df = pd.read_parquet(parquet_path)
    df = df.dropna(subset=['code']).reset_index(drop=True)
    codes = df['code'].tolist()

    preds = []
    for i in tqdm(range(0, len(codes), batch_size), desc="Predicting"):
        batch = codes[i:i + batch_size]
        enc = tokenizer(
            batch, truncation=True, padding=True,
            max_length=max_length, return_tensors="pt",
        )
        logits = model(
            input_ids=enc["input_ids"].to(device),
            attention_mask=enc["attention_mask"].to(device),
        ).logits
        preds.extend(logits.argmax(dim=-1).cpu().tolist())

    df['prediction'] = preds
    return df


def evaluate_by_category(df, tag="GraphCodeBERT"):
    """Print classification metrics for 4 evaluation settings + overall."""
    lang_col = next(
        (c for c in df.columns if c.lower() in ('language', 'lang', 'programming_language')),
        None,
    )
    domain_col = next(
        (c for c in df.columns if c.lower() in ('domain', 'task_type', 'category')),
        None,
    )

    if 'label' not in df.columns:
        print(f"[{tag}] No 'label' column — cannot evaluate.")
        return

    # Overall metrics (always print)
    y_true, y_pred = df['label'].values, df['prediction'].values
    macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    acc = accuracy_score(y_true, y_pred)

    print(f"\n{'='*70}")
    print(f"  TEST RESULTS  --  {tag}")
    print(f"{'='*70}")

    print("\nOverall Classification Report:")
    print(classification_report(y_true, y_pred, zero_division=0))
    print(f"  OVERALL  (n={len(df)})  Accuracy={acc:.4f}  Macro F1={macro_f1:.4f}")

    # Per-category breakdown (if columns exist)
    if lang_col is not None and domain_col is not None:
        _norm = lambda v: str(v).strip().lower()
        df['_l'] = df[lang_col].apply(_norm)
        df['_d'] = df[domain_col].apply(_norm)

        settings = [
            ("(i)   Seen Lang & Seen Domain",      SEEN_LANGS,   SEEN_DOMAINS),
            ("(ii)  Unseen Lang & Seen Domain",    UNSEEN_LANGS, SEEN_DOMAINS),
            ("(iii) Seen Lang & Unseen Domain",    SEEN_LANGS,   UNSEEN_DOMAINS),
            ("(iv)  Unseen Lang & Unseen Domain",  UNSEEN_LANGS, UNSEEN_DOMAINS),
        ]

        for name, langs, domains in settings:
            mask = df['_l'].isin(langs) & df['_d'].isin(domains)
            sub = df[mask]
            n = len(sub)
            if n == 0:
                print(f"\n  {name}:  ** no samples **")
                continue
            y_t, y_p = sub['label'].values, sub['prediction'].values
            sub_acc = accuracy_score(y_t, y_p)
            sub_mf1 = f1_score(y_t, y_p, average='macro', zero_division=0)
            p, r, f1, _ = precision_recall_fscore_support(
                y_t, y_p, average='weighted', zero_division=0
            )
            print(f"\n  {name}  (n={n})")
            print(f"    Accuracy={sub_acc:.4f}  Macro F1={sub_mf1:.4f}")
            print(classification_report(y_t, y_p, zero_division=0))

        df.drop(columns=['_l', '_d'], inplace=True)

    print("=" * 70)

print("evaluate_on_test(), evaluate_by_category() defined.")

evaluate_on_test(), evaluate_by_category() defined.


---
## Run Training + Evaluation

In [6]:
# ============================================================
# Cell 6: Run GraphCodeBERT on Task A
# ============================================================

print("\n" + "=" * 70)
print("  GraphCodeBERT — Task A (Binary Classification)")
print("=" * 70)

trainer_obj = GraphCodeBERTTrainerA(
    model_name=MODEL_NAME,
    max_length=MAX_LENGTH,
)

hf_trainer = trainer_obj.run_full_pipeline(
    output_dir=OUTPUT_DIR,
    num_epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    sample_size=TRAIN_SAMPLE_SIZE,
    val_sample_size=VAL_SAMPLE_SIZE,
    random_seed=RANDOM_SEED,
)


  GraphCodeBERT — Task A (Binary Classification)
[graphcodebert-base] Using LOLO Validation. Holding out: c++
[graphcodebert-base] Train: 49999,  Val: 12499
[graphcodebert-base] Initialising extended model (Multi-Sample Dropout) and tokenizer ...


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

...

Map:   0%|          | 0/12499 [00:00<?, ? examples/s]

[graphcodebert-base] Starting training ...
[graphcodebert-base] Config:
  Train: 49999, Val: 12499
  LR: 2e-05, MaxLen: 512
  Gradient Checkpointing: False (Speed optimization via micro-batching)


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,F1,Precision,Recall
0,0.012400,0.495050,0.646852,0.587970,0.595285,0.754259,0.646852
1,0.036000,1.103669,0.740059,0.718134,0.721826,0.802293,0.740059
2,0.024700,0.894682,0.798064,0.789142,0.791179,0.829233,0.798064
4,0.005400,0.962650,0.826946,0.820782,0.822343,0.852455,0.826946
5,0.011400,0.947540,0.824946,0.818354,0.819979,0.852430,0.824946
6,0.005800,1.150896,0.795504,0.784723,0.786986,0.835475,0.795504
8,0.004300,1.250036,0.787263,0.774641,0.777146,0.833164,0.787263
9,0.003200,1.220409,0.793343,0.781942,0.784284,0.835683,0.793343


[graphcodebert-base] Training completed. Model saved to /kaggle/working/results_graphcodebert_taskA

  EVALUATION — graphcodebert-base



Per-Class Classification Report:
              precision    recall  f1-score   support

       human       0.95      0.67      0.79      5956
     machine       0.76      0.97      0.85      6543

    accuracy                           0.83     12499
   macro avg       0.86      0.82      0.82     12499
weighted avg       0.85      0.83      0.82     12499


>>> COMPETITION METRIC (Macro F1): 0.8208 <<<

[graphcodebert-base] Pipeline completed successfully!


In [7]:
# ============================================================
# Cell 7: Evaluate on test set
# ============================================================

import os
if os.path.exists(TEST_PATH):
    test_df = evaluate_on_test(
        trainer_obj=trainer_obj,
        parquet_path=TEST_PATH,
        max_length=MAX_LENGTH,
        batch_size=BATCH_SIZE * 2,
        device="cuda",
    )
    evaluate_by_category(test_df, tag="GraphCodeBERT-TaskA")
else:
    print(f"Test file not found at {TEST_PATH}. Skipping test evaluation.")

Predicting: 100%|██████████| 32/32 [00:08<00:00,  3.91it/s]


  TEST RESULTS  --  GraphCodeBERT-TaskA

Overall Classification Report:
              precision    recall  f1-score   support

           0       0.71      0.12      0.21       777
           1       0.21      0.82      0.34       223

    accuracy                           0.28      1000
   macro avg       0.46      0.47      0.27      1000
weighted avg       0.60      0.28      0.24      1000

  OVERALL  (n=1000)  Accuracy=0.2790  Macro F1=0.2735


In [8]:
# ============================================================
# Cell 8: Clean up GPU memory
# ============================================================

del hf_trainer, trainer_obj
gc.collect()
torch.cuda.empty_cache()
print("GPU memory freed.")

GPU memory freed.
